In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import config

# 02 - So sanh ket qua 3 model + ablation

In [2]:
import json

import numpy as np
import pandas as pd

from src import evaluate
from src.threshold import load_threshold

pd.set_option("display.float_format", "{:.3f}".format)

# Chỉ lấy model đã có đủ oof / test / threshold; model thiếu file được bỏ qua kèm thông báo.
models = evaluate.available_models()
y_test = evaluate._load_test_labels().to_numpy()
test_proba = {name: np.load(config.test_path(name)) for name in models}
thresholds = {name: load_threshold(name) for name in models}
print("Model co du file:", models)
print("So mau test:", len(y_test), "| so Fail:", int((y_test == config.FAIL).sum()))

Model co du file: ['lr', 'rf', 'xgb']
So mau test: 314 | so Fail: 17


## 1. Model level (tập test, ngưỡng chọn trên OOF của train)

In [3]:
rows = {}
for name in models:
    row = evaluate.model_level(y_test, test_proba[name], thresholds[name])
    row["detection"] = f"{row['n_fail_detected']}/{row['n_fail_total']}"
    for metric in evaluate.BOOTSTRAP_METRICS:
        low, high = evaluate.bootstrap_ci(y_test, test_proba[name], thresholds[name], metric, seed=config.SEED)
        row[f"{metric}_95ci"] = f"[{low:.3f}, {high:.3f}]"
    rows[name] = row
model_table = pd.DataFrame(rows).T
model_table

,threshold,precision,recall,f1,roc_auc,pr_auc,detection_rate,n_fail_total,n_fail_detected,detection,precision_95ci,recall_95ci,f1_95ci,roc_auc_95ci
lr,0.600,0.125,0.294,0.175,0.647,0.184,0.294,17,5,5/17,"[0.031, 0.238]","[0.091, 0.530]","[0.045, 0.313]","[0.500, 0.792]"
rf,0.150,0.091,0.059,0.071,0.634,0.086,0.059,17,1,1/17,"[0.000, 0.333]","[0.000, 0.200]","[0.000, 0.222]","[0.511, 0.762]"
xgb,0.140,0.078,0.294,0.123,0.623,0.097,0.294,17,5,5/17,"[0.016, 0.153]","[0.077, 0.545]","[0.027, 0.225]","[0.485, 0.750]"


## 2. System level (false alarm, latency, kích thước model, robustness)

In [4]:
system_rows = {}
robustness_rows = []
for name in models:
    info = evaluate.system_level(name)
    for scenario in info.pop("robustness", []):
        robustness_rows.append({"model": name, **scenario})
    system_rows[name] = info
system_table = pd.DataFrame(system_rows).T
display(system_table)
if robustness_rows:
    display(pd.DataFrame(robustness_rows))
else:
    print("Chua co robustness_<model>.json (chay python -m src.train --model <ten> --time-block K)")

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.9.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimato

,model,status,threshold,n_flagged,false_alarm,latency_ms_mean,latency_ms_std,model_size_kb
lr,lr,ok,0.600,40,0.875,2.059,6.793,29.481
rf,rf,ok,0.150,11,0.909,31.499,5.392,4008.795
xgb,xgb,ok,0.140,64,0.922,2.446,0.549,422.614


Chua co robustness_<model>.json (chay python -m src.train --model <ten> --time-block K)


## 3. Industrial level (tập test)

So sánh với 3 mốc: không kiểm tra, kiểm tra ngẫu nhiên cùng số lượng (cùng flag_rate), kiểm tra 100%.
Không quy ra tiền; SECOM không đo được downtime và năng lượng nên không đưa 2 chỉ số này vào.

In [5]:
industrial_rows = []
for name in models:
    info = evaluate.industrial_level(y_test, test_proba[name], thresholds[name])
    for plan in ["model", "khong_kiem_tra", "kiem_tra_ngau_nhien", "kiem_tra_100%"]:
        industrial_rows.append({"model": name, "phuong_an": plan, **info[plan]})
industrial_table = pd.DataFrame(industrial_rows).set_index(["model", "phuong_an"])
industrial_table

flag_rate  ti_le_loi_chan_duoc  n_loi_chan_duoc  \
model phuong_an                                                              
lr    model                    0.127                0.294            5.000   
      khong_kiem_tra           0.000                0.000            0.000   
      kiem_tra_ngau_nhien      0.127                0.127            2.166   
      kiem_tra_100%            1.000                1.000           17.000   
rf    model                    0.035                0.059            1.000   
      khong_kiem_tra           0.000                0.000            0.000   
      kiem_tra_ngau_nhien      0.035                0.035            0.596   
      kiem_tra_100%            1.000                1.000           17.000   
xgb   model                    0.204                0.294            5.000   
      khong_kiem_tra           0.000                0.000            0.000   
      kiem_tra_ngau_nhien      0.204                0.204            3.465   
      kiem_tra_100%            1.000                1.000           17.000   

                           n_loi_lot_qua  canh_bao_tren_per_lo  
model phuong_an                                                 
lr    model                       12.000               127.389  
      khong_kiem_tra              17.000                 0.000  
      kiem_tra_ngau_nhien         14.834               127.389  
      kiem_tra_100%                0.000              1000.000  
rf    model                       16.000                35.032  
      khong_kiem_tra              17.000                 0.000  
      kiem_tra_ngau_nhien         16.404                35.032  
      kiem_tra_100%                0.000              1000.000  
xgb   model                       12.000               203.822  
      khong_kiem_tra              17.000                 0.000  
      kiem_tra_ngau_nhien         13.535               203.822  
      kiem_tra_100%                0.000              1000.000

## 4. Capture@k (kiểm tra top k% lô có risk cao nhất trên tập test)

In [6]:
capture_table = pd.concat(
    {name: evaluate.capture_at_k(y_test, test_proba[name], evaluate.CAPTURE_KS) for name in models},
    names=["model"],
).droplevel(1).reset_index()
capture_table.pivot(index="k", columns="model", values=["n_fail_captured", "capture_rate"])

n_fail_captured             capture_rate            
model              lr    rf   xgb           lr    rf   xgb
k                                                         
0.050           3.000 1.000 1.000        0.176 0.059 0.059
0.100           4.000 2.000 1.000        0.235 0.118 0.059
0.200           6.000 6.000 5.000        0.353 0.353 0.294

## 5. Ablation từng model

Số liệu ablation tính trên xác suất OOF của train (không đụng test); ngưỡng của mỗi biến thể chọn theo F1
trên OOF, riêng `threshold_0.5` dùng lại OOF của `baseline` với ngưỡng cố định 0.5.

In [7]:
ablation = {}
for name in config.MODEL_NAMES:
    path = config.ablation_path(name)
    if not path.exists():
        print(f"[{name}] bo qua - chua co {path} (chay python -m src.train --model {name} --ablation)")
        continue
    ablation[name] = pd.DataFrame(json.loads(path.read_text(encoding="utf-8"))).set_index("variant")

for name, table in ablation.items():
    print(f"===== Ablation {name} =====")
    display(table)

===== Ablation lr =====


,threshold,precision,recall,f1,roc_auc,pr_auc
variant,,,,,,
baseline,0.600,0.178,0.471,0.259,0.704,0.159
no_class_weight,0.080,0.155,0.529,0.240,0.697,0.142
no_feature_selection,0.300,0.158,0.368,0.221,0.621,0.125
threshold_0.5,0.500,0.157,0.575,0.247,0.704,0.159


===== Ablation rf =====


,threshold,precision,recall,f1,roc_auc,pr_auc
variant,,,,,,
baseline,0.150,0.226,0.345,0.273,0.723,0.182
no_class_weight,0.210,0.207,0.276,0.236,0.709,0.143
no_feature_selection,0.100,0.184,0.483,0.267,0.717,0.195
threshold_0.5,0.500,0.000,0.000,0.000,0.723,0.182


===== Ablation xgb =====


,threshold,precision,recall,f1,roc_auc,pr_auc
variant,,,,,,
baseline,0.140,0.183,0.391,0.249,0.702,0.159
no_scale_pos_weight,0.030,0.168,0.483,0.249,0.705,0.172
no_feature_selection,0.090,0.252,0.322,0.283,0.739,0.192
threshold_0.5,0.500,0.244,0.126,0.167,0.702,0.159


In [8]:
# So sánh nhanh F1 / PR-AUC của các biến thể giữa các model.
if ablation:
    combined = pd.concat(ablation, names=["model"]).reset_index()
    display(combined.pivot(index="variant", columns="model", values=["f1", "pr_auc"]))

f1             pr_auc            
model                   lr    rf   xgb     lr    rf   xgb
variant                                                  
baseline             0.259 0.273 0.249  0.159 0.182 0.159
no_class_weight      0.240 0.236   NaN  0.142 0.143   NaN
no_feature_selection 0.221 0.267 0.283  0.125 0.195 0.192
no_scale_pos_weight    NaN   NaN 0.249    NaN   NaN 0.172
threshold_0.5        0.247 0.000 0.167  0.159 0.182 0.159